# CSV File Ingestion with TeradataGenAI Vector Store - Complete Lifecycle Management

This notebook demonstrates **end-to-end CSV file ingestion with collection lifecycle management** using the modern **Ingestor fluent API** for comprehensive document processing.

## 🎯 **What This Notebook Accomplishes**
1. **CSV Document Processing** - Extract text, metadata from structured CSV files using Ingestor API
2. **File-Content-Based Collection** - Process sales data from CSV files with runtime embedding generation  
3. **Collection Lifecycle Management** - Create, update, search, and properly destroy collections
4. **Modern Ingestor Pipeline** - Using fluent API pattern following PDF ingestor best practices
5. **Complete Pipeline Demonstrations** - From file ingestion to searchable vector collections

## 📋 **Ingestor API vs REST API Benefits**
- **Before**: Manual REST calls with complex CSV payloads, difficult collection management
"- **Now**: Simple fluent API with `.files()`, `.upsert()`, `.run()` and proper cleanup\n",

## 🔄 **Collection Management Strategy**
- **Cleanup First**: Destroy existing collections before creating new ones to avoid conflicts
- **Resource Management**: Properly clean up collections when no longer needed
- **Naming Convention**: Clear, descriptive collection names that indicate purpose and ingestor type

---

**🚀 Complete CSV document processing pipeline with proper lifecycle management!**

<span style="color: #FFB000;">

## 1. Setup and Imports

Import required libraries for Ingestor workflow and CSV handling

</span>

In [ ]:
# 1. Environment Setup and Auto-Reload Configuration
print("🔧 Setting up CSV file-based vector store environment...")

import sys
import os
import tempfile
from pathlib import Path
import pandas as pd
from datetime import datetime

# Now import the modules that we want to auto-reload
print("📚 Importing TeradataGenAI modules...")

# TeradatamL imports
from teradataml import create_context, remove_context

# TeradataGenAI file-based imports
import teradatagenai
from teradatagenai import (
    Collection, CollectionManager,
    LocalConfig, S3Config, AzureBlobConfig, GCPConfig,
    BasicIngestor, NVIngestor, UnstructuredIngestor,
    ExtractionSchema, ColumnInfo, HNSW, FLAT, SearchParams,
    Ingestor, CollectionType, TeradataAI
)

# Get the directory one level above the current notebook file (notebooks directory)
base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if '__file__' in dir() else os.path.dirname(os.getcwd())

from teradatasqlalchemy.types import VARCHAR, INTEGER, CLOB, VECTOR

print("✅ All imports successful!")
print(f"📂 Base directory: {base_dir}")
print("📚 Environment ready for CSV file-based vector store creation")
print("🔄 Modules will now auto-reload when you modify source code")

🔧 Setting up CSV file-based vector store environment...
📚 Importing TeradataGenAI modules...
✅ All imports successful!
📂 Base directory: c:\Aanchal\office\git\NexusAI\notebooks
📚 Environment ready for CSV file-based vector store creation
🔄 Modules will now auto-reload when you modify source code


<span style="color: #FFB000;">

## 2. Secure Credential Collection

Collect all credentials securely using getpass - NOT displayed in outputs

</span>

In [ ]:
# 2. Secure Credential Collection
print("🔐 Collecting credentials securely...")

import os
from getpass import getpass

# Database credentials
base_url = getpass("Enter TD Base URL (e.g., https://aiop-btms4.td.teradata.com): ")
os.environ['TD_HOST'] = getpass("Enter TD_HOST: ")
os.environ['TD_USERNAME'] = getpass("Enter TD_USERNAME: ")
os.environ['TD_PASSWORD'] = getpass("Enter TD_PASSWORD: ")
os.environ['TD_BASE_URL'] = base_url
os.environ['TD_AUTH_MECH'] = 'BASIC'
os.environ['NVIDIA_API_KEY'] = getpass("Enter NVIDIA_API_KEY: ")

# ML model/API credentials
embeddings_base_url = getpass("Enter Embeddings Base URL: ")
nvidia_api_key = os.environ['NVIDIA_API_KEY']

# AWS S3 credentials
aws_access_key_id = getpass("Enter AWS Access Key ID: ")
aws_secret_access_key = getpass("Enter AWS Secret Access Key: ")
aws_session_token = getpass("Enter AWS Session Token: ")
s3_bucket_name = getpass("Enter S3 Bucket Name: ")
s3_region_name = getpass("Enter S3 Region Name (e.g., us-west-2): ")

# Azure credentials
azure_account_name = getpass("Enter Azure Account Name: ")
azure_account_key = getpass("Enter Azure Account Key: ")
azure_container_name = getpass("Enter Azure Container Name: ")

# GCP credentials
gcp_bucket_name = getpass("Enter GCP Bucket Name: ")
gcp_project_id = getpass("Enter GCP Project ID: ")
gcp_client_email = getpass("Enter GCP Client Email: ")
print("Enter GCP Secret JSON (paste the full JSON, then press Enter):")
import json
gcp_secret_str = getpass("GCP Secret JSON: ")
gcp_secret = json.loads(gcp_secret_str)

# Store variables for later use
TD_HOST = os.environ['TD_HOST']
TD_USER = os.environ['TD_USERNAME']
TD_PASSWORD = os.environ['TD_PASSWORD']

print("✅ All credentials collected securely")
print(f"🌐 Host: {TD_HOST}")
print(f"👤 User: {TD_USER}")
print("🔒 Credentials stored securely (not displayed)")
print("☁️ Multi-cloud credentials configured for S3, Azure Blob, and GCP")

In [4]:
# 2. Database Connection Configuration
print("🔗 Configuring database connection...")

# Create database context
create_context()

print("✅ Database connection established")
print(f"🌐 Connected to: {os.environ['TD_HOST']}")
print(f"👤 User: {os.environ['TD_USERNAME']}")

# Verify service health
try:
    health = CollectionManager.health()
    print("✅ Collection service is healthy")
except Exception as e:
    print(f"ℹ️ Service status: {e}")

🔗 Configuring database connection...
Authentication token is generated and set for the session.
✅ Database connection established
🌐 Connected to: 10.27.178.11
👤 User: oaf
✅ Collection service is healthy


<span style="color: #FFB000;">

## 3. Configure CSV Data (Content Only)

Setup CSV files WITHOUT embeddings for FILE-CONTENT-BASED ingestion

</span>

In [5]:
# 3. Configure Local CSV File Ingestion - The Modern Way
print("📁 Setting up Local CSV File Ingestion with Ingestor API...")

# Use the specific CSV file path for chocolate sales data
csv_file_path = os.path.join(base_dir, 'example-data', 'csv', 'Chocolate_Sales.csv')
csv_file_path_extra = os.path.join(base_dir, 'example-data', 'csv', 'Chocolate_Sales_new.csv')

# Verify file exists
if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"CSV file not found at: {csv_file_path}")

print(f"✅ CSV file found: {csv_file_path}")

# Load and inspect the CSV structure
csv_data = pd.read_csv(csv_file_path)

print(f"✓ CSV file loaded successfully")
print(f"✓ Number of rows: {len(csv_data)}")
print(f"✓ Columns: {list(csv_data.columns)}")
print(f"✓ Sample row: {csv_data.head(1).to_dict('records')[0] if len(csv_data) > 0 else 'No data'}")

# Display sample for verification
print("\n📊 Sample data:")
print(csv_data.head())

# Define collection names with clear, descriptive identifiers  
csv_collection_name = "csv_chocolate_sales_content_ingestor"

# Configure embedding model (same as validation examples)
csv_embedding_model = TeradataAI(
    api_type="nim",
    model_name="nvidia/llama-3.2-nv-embedqa-1b-v2",
    api_base=embeddings_base_url
)

# Create LocalConfig for CSV file from specified path
csv_local_config = LocalConfig(
    files=[csv_file_path],  # Use the specific file path
    files_type="csv",
    delimiter=",",  # Standard CSV delimiter
)

print(f"\n✅ LocalConfig created for CSV file ingestion")
print(f"✓ File path: {csv_file_path}")
print(f"✓ File type: csv")
print(f"✓ Embedding model: {csv_embedding_model.model_name}")

📁 Setting up Local CSV File Ingestion with Ingestor API...
✅ CSV file found: c:\Aanchal\office\git\NexusAI\notebooks\example-data\csv\Chocolate_Sales.csv
✓ CSV file loaded successfully
✓ Number of rows: 3240
✓ Columns: ['Sales Person', 'Country', 'Product', 'Date', 'Amount', 'Boxes Shipped']
✓ Sample row: {'Sales Person': 'Jehu Rudeforth', 'Country': 'UK', 'Product': 'Mint Chip Choco', 'Date': '04/01/2022', 'Amount': '$5,320.00', 'Boxes Shipped': 180}

📊 Sample data:
     Sales Person    Country              Product        Date      Amount  \
0  Jehu Rudeforth         UK      Mint Chip Choco  04/01/2022   $5,320.00   
1     Van Tuxwell      India        85% Dark Bars  01/08/2022   $7,896.00   
2    Gigi Bohling      India  Peanut Butter Cubes  07/07/2022   $4,501.00   
3    Jan Morforth  Australia  Peanut Butter Cubes  27/04/2022  $12,726.00   
4  Jehu Rudeforth         UK  Peanut Butter Cubes  24/02/2022  $13,685.00   

   Boxes Shipped  
0            180  
1             94  
2       

<span style="color: #FFB000;">

## 4. Configure Embedding Model and Collection Settings

Set up TeradataAI embedding model for runtime embedding generation

</span>

In [6]:
# 4. Configure Extraction Schema based on CSV structure for chocolate sales data
print("📋 Configuring extraction schema for chocolate sales CSV...")

# Configure extraction schema for chocolate sales data processing
csv_extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(
            name="Country",
            datatype=VARCHAR(100),
            description="Country where the sale occurred"
        ),
        ColumnInfo(
            name="Product",
            datatype=VARCHAR(200),
            description="Chocolate product name"
        ),
        ColumnInfo(
            name="Amount",
            datatype=VARCHAR(500),
            description="Sale amount"
        )
    ]
)

print("✅ Extraction schema configured for chocolate sales CSV")
print(f"  Table name: csv_chocolate_sales_content")
print(f"  Data columns: Sales Person, Country, Product, Date, Amount, Boxes Shipped")
print(f"  Collection type: FILE-CONTENT-BASED")
print(f"  Embedding generation: Runtime using {csv_embedding_model.model_name}")

📋 Configuring extraction schema for chocolate sales CSV...
✅ Extraction schema configured for chocolate sales CSV
  Table name: csv_chocolate_sales_content
  Data columns: Sales Person, Country, Product, Date, Amount, Boxes Shipped
  Collection type: FILE-CONTENT-BASED
  Embedding generation: Runtime using nvidia/llama-3.2-nv-embedqa-1b-v2


---

## Test 1: FILE-CONTENT-BASED Ingestion

**Objective:** Test CSV file ingestion with runtime embedding generation

**Expected Behavior:**
1. Upload CSV files → Vector Store
2. Service extracts structured data from CSV
3. Service generates embeddings using configured model
4. Data loaded into tables with searchable content

**What Should Happen:**
- ✅ Files upload successfully
- ✅ CSV parsing extracts sales data
- ✅ Embeddings generated for semantic search
- ✅ Data tables created with sales information

**Proof Point:** This workflow enables semantic search over chocolate sales data

In [7]:
# 5. Create and Execute CSV File Ingestion Pipeline
print("🚀 Creating CSV file ingestion pipeline...")

# Clean up existing collection if it exists
try:
    existing_collection = Collection(name=csv_collection_name)
    existing_collection.destroy()
    print(f"✅ Cleaned up existing collection: {csv_collection_name}")
except Exception as e:
    print(f"ℹ️ No existing collection to clean up: {e}")

print("\n" + "="*80)
print("🔌 CSV FILE INGESTION WITH INGESTOR API")
print("="*80)
print("\nFollowing PDF ingestor pattern with CSV file processing")
print(f"Source file: {csv_file_path}")
print("-"*80)

try:
    # Build Ingestor pipeline using fluent API (same pattern as PDF ingestor)
    csv_ingestor_pipeline = (
        Ingestor(
            name=csv_collection_name,
            type=CollectionType.FILE_CONTENT_BASED,
            target_database=TD_USER,
            description="CSV chocolate sales data ingestion using Ingestor API"
        )
        .files(
            files=csv_local_config,
            extraction_schema=csv_extraction_schema
        )
        .upsert(embedding_model=csv_embedding_model)
    )
    
    print("✅ Ingestor pipeline created successfully")
    print(f"  Collection: {csv_collection_name}")
    print(f"  Type: FILE-CONTENT-BASED")
    print(f"  File: {csv_file_path}")
    print(f"  Embedding Model: {csv_embedding_model.model_name}")
    
    print("\n🚀 Executing CSV ingestion pipeline...")
    result = csv_ingestor_pipeline.run()
    
    success = result.get('status', {}).get('success', False)
    
    if success:
        print("\n✅ CSV INGESTION SUCCESS!")
        print(f"  Collection: {csv_collection_name}")
        print(f"  Status: CSV file processed and embeddings generated")
            
    else:
        print(f"\n❌ CSV INGESTION FAILED")
        print(f"  Status: {result.get('status', {})}")
        
        try:
            collection_check = Collection(name=csv_collection_name)
            status_data = collection_check.status(return_type="json")
            print(f"\n  Collection Status Details:")
            print(f"    Status: {status_data.get('collection_status')}")
            if 'error_message' in status_data:
                print(f"    Error: {status_data['error_message']}")
        except Exception as status_error:
            print(f"  Could not get collection status: {status_error}")
        
except Exception as e:
    print(f"\n❌ INGESTION FAILED WITH EXCEPTION")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

🚀 Creating CSV file ingestion pipeline...


C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection does not exist or it has been destroyed successfully.
✅ Cleaned up existing collection: csv_chocolate_sales_content_ingestor

🔌 CSV FILE INGESTION WITH INGESTOR API

Following PDF ingestor pattern with CSV file processing
Source file: c:\Aanchal\office\git\NexusAI\notebooks\example-data\csv\Chocolate_Sales.csv
--------------------------------------------------------------------------------
✅ Ingestor pipeline created successfully
  Collection: csv_chocolate_sales_content_ingestor
  Type: FILE-CONTENT-BASED
  File: c:\Aanchal\office\git\NexusAI\notebooks\example-data\csv\Chocolate_Sales.csv
  Embedding Model: nvidia/llama-3.2-nv-embedqa-1b-v2

🚀 Executing CSV ingestion pipeline...
Initializing collection pipeline...


C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/10

In [8]:
result

{'status': {'success': True,
  'errors': [],
  'warnings': ['No ingestor specified; BasicIngestor() will be used by default'],
  'message': 'Pipeline executed successfully'},
 'collection': 'csv_chocolate_sales_content_ingestor',
 'operation_details': {'create_collection': {'success': True,
   'api_response': 'Collection initialized successfully',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_chocolate_sales_content_ingestor',
     'collection_status': 'initialized'}}},
  'ingest': {'success': True,
   'api_response': 'Files uploaded successfully and ingestion in progress',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_chocolate_sales_content_ingestor',
     'collection_status': 'initialized'},
    'warnings': ['Operation completed but success status unclear']}},
  'create_index': {'success': True,
   'api_response': None,
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv

In [9]:
# Test similarity search to verify ingestion worked
try:
    print("\n🔍 Verification: Testing similarity search on chocolate sales data...")
    collection = Collection(name=csv_collection_name)
    
    # Test search for chocolate products
    search_results = collection.similarity_search(
        question="What chocolate products are available?",
        top_k=5
    )
    
    print("Search results for chocolate products:")
    print(search_results)
        
except Exception as search_error:
    print(f"  ⚠️ Search verification failed: {search_error}")


🔍 Verification: Testing similarity search on chocolate sales data...
Collection csv_chocolate_sales_content_ingestor initialized for use.
Search results for chocolate products:
similar_objects_count:5
similar_objects:
      score DataBaseName                                      TableName  TD_ID          td_filename AttributeName AttributeValue  index_label
0  0.733912          oaf  ingest_table_da65a01ae2db4bae8b1035cb652d72c7   1743  Chocolate_Sales.csv       product     White Choc            2
1  0.733912          oaf  ingest_table_da65a01ae2db4bae8b1035cb652d72c7   1740  Chocolate_Sales.csv       product     White Choc            4
2  0.733912          oaf  ingest_table_da65a01ae2db4bae8b1035cb652d72c7   1731  Chocolate_Sales.csv       product     White Choc            3
3  0.733912          oaf  ingest_table_da65a01ae2db4bae8b1035cb652d72c7   1734  Chocolate_Sales.csv       product     White Choc            1
4  0.733912          oaf  ingest_table_da65a01ae2db4bae8b1035cb652d72c7

In [10]:
# Test search for UK sales information
try:
    search_results2 = collection.similarity_search(
        question="Sales in UK market",
        top_k=3
    )
    
    print("Search results for UK sales:")
    print(search_results2)
        
except Exception as search_error:
    print(f"  ⚠️ Search verification failed: {search_error}")

Search results for UK sales:
similar_objects_count:3
similar_objects:
      score DataBaseName                                      TableName  TD_ID          td_filename AttributeName AttributeValue  index_label
0  0.805511          oaf  ingest_table_da65a01ae2db4bae8b1035cb652d72c7   6542  Chocolate_Sales.csv       country             UK            2
1  0.805511          oaf  ingest_table_da65a01ae2db4bae8b1035cb652d72c7   6548  Chocolate_Sales.csv       country             UK            1
2  0.805511          oaf  ingest_table_da65a01ae2db4bae8b1035cb652d72c7   6551  Chocolate_Sales.csv       country             UK            0)


---

## ☁️ Part 2: Multi-Cloud CSV Storage Integration

**Use Case:** Ingest CSV files from cloud storage (S3, Azure Blob, GCP) with structured data processing  
**Key Feature:** Multi-cloud CSV integration with BasicIngestor for structured sales data

### 📊 CSV Cloud Processing Capabilities:
- **Cloud Storage**: S3Config, AzureBlobConfig, GCPConfig for CSV files with sales data
- **Structured Data**: Process CSV files with columns like product, sales, region, etc.
- **Content-Based Processing**: Extract and embed CSV row data for similarity search
- **Collection Management**: Proper cleanup and lifecycle management for CSV collections
- **Search Capabilities**: Find similar sales data, products, or regional information

**🚀 Complete CSV cloud processing pipeline from multiple providers!**

In [11]:
# Configure S3 CSV File Ingestion for Sales Analytics
print("📊 Setting up S3 CSV File Ingestion for Chocolate Sales Data...")

# Define CSV cloud storage collection names
s3_csv_collection_name = "csv_s3_chocolate_sales_content"
azure_csv_collection_name = "csv_azure_chocolate_sales_content"  
gcp_csv_collection_name = "csv_gcp_chocolate_sales_content"

# Configure embedding model (same as local configuration)
csv_embedding_model = TeradataAI(
    api_type="nim",
    model_name="nvidia/llama-3.2-nv-embedqa-1b-v2",
    api_base=embeddings_base_url
)

# S3 configuration for CSV file processing with credentials from getpass
s3_csv_config = S3Config(
    bucket=s3_bucket_name,
    key="csv-data/Chocolate_Sales.csv",
    region_name=s3_region_name,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,
    files_type="csv",
    delimiter=","
)

# Configure extraction schema for S3 CSV content
s3_csv_extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="Country", datatype=VARCHAR(100), description="Country where the sale occurred"),
        ColumnInfo(name="Product", datatype=VARCHAR(200), description="Chocolate product name"),
        ColumnInfo(name="Amount", datatype=VARCHAR(500), description="Sale amount")
    ]
)

print("✅ S3 CSV configuration complete!")
print(f"📂 S3 bucket: {s3_csv_config.bucket}")
print(f"📊 CSV file: {s3_csv_config.key}")
print(f"🌐 S3 region: {s3_csv_config.region_name}")
print("🔄 Ready for S3 CSV ingestion with structured data processing")


📊 Setting up S3 CSV File Ingestion for Chocolate Sales Data...
✅ S3 CSV configuration complete!
📂 S3 bucket: genaitestaanchal
📊 CSV file: csv-data/Chocolate_Sales.csv
🌐 S3 region: us-west-2
🔄 Ready for S3 CSV ingestion with structured data processing


In [12]:
# Clean up existing S3 CSV collection before creating new one
print(f"🧹 Cleaning up existing S3 CSV collection: {s3_csv_collection_name}")
try:
    Collection(name=s3_csv_collection_name).destroy()
    print("✅ Existing S3 CSV collection destroyed")
except Exception as e:
    print(f"ℹ️ No existing S3 CSV collection to destroy: {e}")

# Create S3 CSV Collection Pipeline
print("🚀 Creating S3 CSV Collection with BasicIngestor for Sales Data...")

try:
    s3_csv_pipeline = (
        Ingestor(
            name=s3_csv_collection_name,
            type=CollectionType.FILE_CONTENT_BASED,
            target_database=TD_USER,
            description="Demo: S3 CSV chocolate sales data ingestion with BasicIngestor"
        )
        .files(
            files=s3_csv_config,
            extraction_schema=s3_csv_extraction_schema
        )
        .upsert(
            embedding_model=csv_embedding_model,
            indexing_algorithm=HNSW(
                metric="COSINE",
                ef_construction=200,
                num_connpernode=32)
        ).run()
    )
    
    print("✅ S3 CSV collection created successfully!")
    print(f"🎯 Pipeline success: {s3_csv_pipeline['status']['success']}")
    
    if s3_csv_pipeline['status']['success']:
        s3_csv_collection = s3_csv_pipeline["collection"]
        print(f"📚 S3 CSV Collection name: {s3_csv_collection.name}")
        print("🎉 S3 CSV sales data processed and indexed!")
    else:
        print(f"⚠️ S3 CSV Pipeline issues: {s3_csv_pipeline['status']['errors']}")
    
except Exception as e:
    print(f"ℹ️ S3 CSV collection creation: {e}")
    s3_csv_collection = Collection(name=s3_csv_collection_name)

print("\n📊 S3 CSV Processing Complete:")
print("  1️⃣ Downloaded CSV from S3 bucket")
print("  2️⃣ Processed structured sales data with BasicIngestor")
print("  3️⃣ Generated embeddings for product and sales information")
print("  4️⃣ Created searchable index for sales data queries")
print("  🎯 Cloud CSV integration with structured data successful!")

🧹 Cleaning up existing S3 CSV collection: csv_s3_chocolate_sales_content


C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection does not exist or it has been destroyed successfully.
✅ Existing S3 CSV collection destroyed
🚀 Creating S3 CSV Collection with BasicIngestor for Sales Data...
Initializing collection pipeline...


C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'csv_s3_chocolate_sales_content' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                            
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

In [13]:
s3_csv_pipeline

{'status': {'success': True,
  'errors': [],
  'warnings': ['No ingestor specified; BasicIngestor() will be used by default'],
  'message': 'Pipeline executed successfully'},
 'collection': 'csv_s3_chocolate_sales_content',
 'operation_details': {'create_collection': {'success': True,
   'api_response': 'Collection initialized successfully',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_s3_chocolate_sales_content',
     'collection_status': 'initialized'}}},
  'ingest': {'success': True,
   'api_response': "File ingestion from remote storage for collection 'csv_s3_chocolate_sales_content' has been scheduled.",
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_s3_chocolate_sales_content',
     'collection_status': 'initialized'},
    'warnings': ['Operation completed but success status unclear']}},
  'create_index': {'success': True,
   'api_response': None,
   'status_result': {'success': True,
    'status_resp

## Azure Blob Storage CSV Integration with Sales Analytics

In [18]:
# Clean up existing Azure CSV collection
print(f"🧹 Cleaning up existing Azure CSV collection: {azure_csv_collection_name}")
try:
    Collection(name=azure_csv_collection_name).destroy()
    print("✅ Existing Azure CSV collection destroyed")
except Exception as e:
    print(f"ℹ️ No existing Azure CSV collection to destroy: {e}")

# Create Azure Blob CSV Collection Pipeline
print("🚀 Creating Azure Blob CSV Collection for Sales Analytics...")

# Configure extraction schema for Azure CSV content
azure_csv_extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="Country", datatype=VARCHAR(100), description="Country where the sale occurred"),
        ColumnInfo(name="Product", datatype=VARCHAR(200), description="Chocolate product name"),
        ColumnInfo(name="Amount", datatype=VARCHAR(500), description="Sale amount")
    ]
)

# Configure Azure Blob storage with credentials from getpass
azure_csv_config = AzureBlobConfig(
    container=azure_container_name,
    blob_name="csv-data/Chocolate_Sales.csv",
    account_name=azure_account_name,
    account_key=azure_account_key,
    files_type="csv",
    delimiter=","
)

try:
    azure_csv_pipeline = (
        Ingestor(
            name=azure_csv_collection_name,
            type=CollectionType.FILE_CONTENT_BASED,
            target_database=TD_USER,
            description="Demo: Azure Blob CSV chocolate sales data ingestion"
        )
        .files(
            files=azure_csv_config,
            extraction_schema=azure_csv_extraction_schema
        )
        .upsert(
            indexing_algorithm=HNSW(
                metric="COSINE",
                ef_construction=200,
                num_connpernode=32),
            embedding_model=csv_embedding_model,
        ).run()
    )
    
    print("✅ Azure Blob CSV collection created successfully!")
    print(f"🎯 Pipeline success: {azure_csv_pipeline['status']['success']}")
    
    if azure_csv_pipeline['status']['success']:
        azure_csv_collection = azure_csv_pipeline["collection"]
        print(f"📚 Azure CSV Collection name: {azure_csv_collection.name}")
        print("🎉 Azure Blob CSV sales data processed and indexed!")
    else:
        print(f"⚠️ Azure CSV Pipeline issues: {azure_csv_pipeline['status']['errors']}")
    
except Exception as e:
    print(f"ℹ️ Azure CSV collection creation: {e}")
    azure_csv_collection = Collection(name=azure_csv_collection_name)

print("\n📊 Azure CSV Processing Complete:")
print("  1️⃣ Downloaded CSV from Azure Blob storage")
print("  2️⃣ Processed structured sales data with BasicIngestor and HNSW indexing")
print("  🎯 Multi-cloud CSV processing with Azure integration!")
print("  3️⃣ Generated embeddings for business analytics queries")

🧹 Cleaning up existing Azure CSV collection: csv_azure_chocolate_sales_content
Collection csv_azure_chocolate_sales_content initialized for use.
Collection destroy request is accepted and in progress
✅ Existing Azure CSV collection destroyed
🚀 Creating Azure Blob CSV Collection for Sales Analytics...
Initializing collection pipeline...
Collection csv_azure_chocolate_sales_content initialized for use.
Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'csv_azure_chocolate_sales_content' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Com

In [19]:
azure_csv_pipeline

{'status': {'success': True,
  'errors': [],
  'warnings': ['No ingestor specified; BasicIngestor() will be used by default'],
  'message': 'Pipeline executed successfully'},
 'collection': 'csv_azure_chocolate_sales_content',
 'operation_details': {'create_collection': {'success': True,
   'api_response': 'Collection initialized successfully',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_azure_chocolate_sales_content',
     'collection_status': 'initialized'}}},
  'ingest': {'success': True,
   'api_response': "File ingestion from remote storage for collection 'csv_azure_chocolate_sales_content' has been scheduled.",
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_azure_chocolate_sales_content',
     'collection_status': 'initialized'},
    'warnings': ['Operation completed but success status unclear']}},
  'create_index': {'success': True,
   'api_response': None,
   'status_result': {'success': True,
    

## Google Cloud Storage CSV Integration with Business Intelligence

In [20]:
# Configure Google Cloud Storage CSV Integration
print("☁️ Setting up Google Cloud Storage CSV configuration for Sales Analytics...")

# Clean up existing GCP CSV collection
print(f"🧹 Cleaning up existing GCP CSV collection: {gcp_csv_collection_name}")
try:
    Collection(name=gcp_csv_collection_name).destroy()
    print("✅ Existing GCP CSV collection destroyed")
except Exception as e:
    print(f"ℹ️ No existing GCP CSV collection to destroy: {e}")

# Configure GCP storage with credentials from getpass
gcp_csv_config = GCPConfig(
    files_type="csv",
    bucket=gcp_bucket_name,
    blob_name="csv-data/Chocolate_Sales.csv",
    project_id=gcp_project_id,
    delimiter=",",
    secret=gcp_secret
)

# Configure extraction schema for GCP CSV content
gcp_csv_extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="Country", datatype=VARCHAR(100), description="Country where the sale occurred"),
        ColumnInfo(name="Product", datatype=VARCHAR(200), description="Chocolate product name"),
        ColumnInfo(name="Amount", datatype=VARCHAR(500), description="Sale amount")
    ]
)

print("✅ Google Cloud Storage CSV configuration complete!")
print(f"📂 GCP bucket: {gcp_csv_config.bucket}")
print(f"📊 CSV file: {gcp_csv_config.blob_name}")
print("🔄 Ready for GCP CSV sales data ingestion")
print(f"🏢 Project: {gcp_csv_config.project_id}")

☁️ Setting up Google Cloud Storage CSV configuration for Sales Analytics...
🧹 Cleaning up existing GCP CSV collection: csv_gcp_chocolate_sales_content


C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection does not exist or it has been destroyed successfully.
✅ Existing GCP CSV collection destroyed
✅ Google Cloud Storage CSV configuration complete!
📂 GCP bucket: ak250090-filestore
📊 CSV file: csv-data/Chocolate_Sales.csv
🔄 Ready for GCP CSV sales data ingestion
🏢 Project: icgcp-vector-store


In [21]:
# Create Google Cloud Storage CSV Collection Pipeline
print("🚀 Creating Google Cloud Storage CSV Collection for Business Intelligence...")

try:
    gcp_csv_pipeline = (
        Ingestor(
            name=gcp_csv_collection_name,
            type=CollectionType.FILE_CONTENT_BASED,
            target_database=TD_USER,
            description="Demo: GCP Storage CSV chocolate sales data ingestion"
        )
        .files(
            files=gcp_csv_config,
            extraction_schema=gcp_csv_extraction_schema
        )
        .upsert(
            indexing_algorithm=HNSW(
                metric="COSINE",
                ef_construction=200,
                num_connpernode=32),
            embedding_model=csv_embedding_model,
        ).run()
    )
    
    print("✅ GCP Storage CSV collection created successfully!")
    print(f"🎯 Pipeline success: {gcp_csv_pipeline['status']['success']}")
    
    if gcp_csv_pipeline['status']['success']:
        gcp_csv_collection = gcp_csv_pipeline["collection"]
        print(f"📚 GCP CSV Collection name: {gcp_csv_collection.name}")
        print("🎉 GCP Storage CSV sales data processed and indexed!")
    else:
        print(f"⚠️ GCP CSV Pipeline issues: {gcp_csv_pipeline['status']['errors']}")
    
except Exception as e:
    print(f"ℹ️ GCP CSV collection creation: {e}")
    gcp_csv_collection = Collection(name=gcp_csv_collection_name)

print("\n📊 GCP CSV Processing Complete:")
print("  1️⃣ Downloaded CSV from Google Cloud Storage")
print("  2️⃣ Processed structured sales data with BasicIngestor and HNSW indexing")
print("  3️⃣ Generated embeddings for comprehensive business analytics")
print("  🎯 Complete multi-cloud CSV processing pipeline with all three providers!")

🚀 Creating Google Cloud Storage CSV Collection for Business Intelligence...
Initializing collection pipeline...


C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'csv_gcp_chocolate_sales_content' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

In [22]:
gcp_csv_pipeline

{'status': {'success': True,
  'errors': [],
  'warnings': ['No ingestor specified; BasicIngestor() will be used by default'],
  'message': 'Pipeline executed successfully'},
 'collection': 'csv_gcp_chocolate_sales_content',
 'operation_details': {'create_collection': {'success': True,
   'api_response': 'Collection initialized successfully',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_gcp_chocolate_sales_content',
     'collection_status': 'initialized'}}},
  'ingest': {'success': True,
   'api_response': "File ingestion from remote storage for collection 'csv_gcp_chocolate_sales_content' has been scheduled.",
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_gcp_chocolate_sales_content',
     'collection_status': 'initialized'},
    'warnings': ['Operation completed but success status unclear']}},
  'create_index': {'success': True,
   'api_response': None,
   'status_result': {'success': True,
    'status_

In [23]:
# Comprehensive Multi-Cloud CSV Testing and Validation
print("\n🔍 Testing All CSV Collections Across Multi-Cloud Infrastructure...")

# Define all CSV collections for comprehensive testing
all_csv_collections = [
    (s3_csv_collection_name, "AWS S3"),
    (azure_csv_collection_name, "Azure Blob"),
    (gcp_csv_collection_name, "Google Cloud")
]

# Business intelligence queries for sales analytics
business_queries = [
    "What are the top performing chocolate products?",
    "Show me sales data by geographic region",
    "Which chocolate types have seasonal trends?",
    "What is the revenue performance analysis?"
]

print("\n📊 Multi-Cloud CSV Business Intelligence Testing:")
print("=" * 60)

for collection_name, cloud_provider in all_csv_collections:
    print(f"\n🌐 Testing {cloud_provider} CSV Collection: {collection_name}")
    print("-" * 50)
    
    try:
        collection_obj = Collection(name=collection_name)
        status = collection_obj.status(return_type="json")
        print(f"   ✅ Collection Status: {status.get('status', 'Unknown')}")
        
        for query in business_queries[:2]:
            try:
                search_result = collection_obj.similarity_search(
                    question=query,
                    search_params=SearchParams(top_k=2)
                )
                print(f"   🎯 Query: '{query}' - Success")
                print(search_result)
                
            except Exception as e:
                print(f"   ⚠️ Query: '{query}' - {str(e)[:50]}...")
                
    except Exception as e:
        print(f"   ❌ Collection Error: {str(e)[:50]}...")

print(f"\n🎯 MULTI-CLOUD CSV INTEGRATION SUMMARY:")
print("✅ S3, Azure Blob, and GCP CSV collections created")
print("✅ BasicIngestor processing for structured sales data")
print("✅ HNSW indexing for fast similarity search")
print("✅ Business intelligence queries supported")
print("✅ Complete cloud provider coverage for CSV ingestion")
print("\n🔄 All CSV collections ready for production business analytics!")


🔍 Testing All CSV Collections Across Multi-Cloud Infrastructure...

📊 Multi-Cloud CSV Business Intelligence Testing:

🌐 Testing AWS S3 CSV Collection: csv_s3_chocolate_sales_content
--------------------------------------------------
Collection csv_s3_chocolate_sales_content initialized for use.
   ✅ Collection Status: Unknown
   🎯 Query: 'What are the top performing chocolate products?' - Success
similar_objects_count:2
similar_objects:
      score DataBaseName                                      TableName  TD_ID          td_filename AttributeName AttributeValue  index_label
0  0.650185          oaf  ingest_table_7ab51f9b86c441a49b2a642939467e25   3735  Chocolate_Sales.csv       product   Almond Choco            1
1  0.650185          oaf  ingest_table_7ab51f9b86c441a49b2a642939467e25    471  Chocolate_Sales.csv       product   Almond Choco            0)
   🎯 Query: 'Show me sales data by geographic region' - Success
similar_objects_count:2
similar_objects:
      score DataBaseName  

## Section 8: One-Shot Ingestor (from_documents/add_documents/delete_documents)

Simple example demonstrating Collection.from_documents(), add_documents(), and delete_documents() with CSV data.

In [24]:
# 8.1: Setup - Document Management Demo
print("📊 CSV Document Management Scenario\n")

doc_mgmt_collection = "csv_doc_mgmt_demo"

# Clean up existing collection
try:
    Collection(name=doc_mgmt_collection).destroy()
except:
    pass

# Setup extraction schema
doc_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="Country", datatype=VARCHAR(100)),
        ColumnInfo(name="Product", datatype=VARCHAR(200)),
        ColumnInfo(name="Amount", datatype=VARCHAR(500))
    ]
)

print("✅ Setup complete - Ready for document management operations\n")

📊 CSV Document Management Scenario



C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection does not exist or it has been destroyed successfully.
✅ Setup complete - Ready for document management operations



In [25]:
# 8.2: Create Collection using from_documents()
print("1️⃣ Creating collection with from_documents()...")
try:
    doc_collection = Collection.from_documents(
        name=doc_mgmt_collection,
        documents=LocalConfig(files=[csv_file_path], files_type="csv"),
        type=CollectionType.FILE_CONTENT_BASED,
        embedding=csv_embedding_model,
        extraction_schema=doc_schema,
        target_database=TD_USER,
        description="CSV document management demo"
    )
    print("   ✅ Collection created\n")
except Exception as e:
    print(f"   ⚠️ Error: {e}\n")
    doc_collection = Collection(name=doc_mgmt_collection)

1️⃣ Creating collection with from_documents()...
Initializing collection pipeline...


C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/10

In [26]:
# 8.3: Add Documents using add_documents()
print("2️⃣ Adding documents with add_documents()...")
try:
    doc_collection.add_documents(
        documents=LocalConfig(files=[csv_file_path_extra], files_type="csv"),
        embedding=csv_embedding_model,
        extraction_schema=doc_schema,
        target_database=TD_USER
    )
    print("   ✅ Documents added\n")
except Exception as e:
    print(f"   ⚠️ Error: {e}\n")

2️⃣ Adding documents with add_documents()...
Initializing collection pipeline...
Collection csv_doc_mgmt_demo initialized for use.
Creating collection...
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Update completed successfully
Pipeline completed successfully! Operations: create_collection, ingest, create_index
   ✅ Documents added



In [27]:
# 8.4: Test with Similarity Search
print("3️⃣ Testing with similarity search...")
try:
    results = doc_collection.similarity_search(
        question="Show chocolate sales in Brazil",
        search_params=SearchParams(top_k=2)
    )
    print("   ✅ Search successful\n")
except Exception as e:
    print(f"   ⚠️ Error: {e}\n")

3️⃣ Testing with similarity search...
   ✅ Search successful



In [28]:
results

similar_objects_count:2
similar_objects:
      score DataBaseName                                      TableName  TD_ID          td_filename AttributeName AttributeValue  index_label
0  0.636916          oaf  ingest_table_931d723b5f984544b007a08bc25c14ba    465  Chocolate_Sales.csv       product   Almond Choco            1
1  0.636916          oaf  ingest_table_931d723b5f984544b007a08bc25c14ba   2169  Chocolate_Sales.csv       product   Almond Choco            0)

In [29]:
# 8.5: Delete Documents using delete_documents()
print("4️⃣ Deleting documents with delete_documents()...")
try:
    doc_collection.delete_documents(documents=["Chocolate_Sales.csv"])
    print("   ✅ Documents deleted\n")
except Exception as e:
    print(f"   ⚠️ Error: {e}\n")

4️⃣ Deleting documents with delete_documents()...
Collection update request is accepted and in progress
   ✅ Documents deleted



## Section 8: Cleanup

In [ ]:
# 7. Comprehensive CSV Collection Cleanup (Local and Cloud)
print("🧹 Comprehensive CSV Collection Cleanup Management...")
print("🔄 This includes local and all cloud storage collections\n")

# List all CSV collections created in this notebook
all_csv_collections = [
    csv_collection_name,
    s3_csv_collection_name,
    azure_csv_collection_name,
    gcp_csv_collection_name,
    doc_mgmt_collection
]

cleanup_results = []

for collection_name in all_csv_collections:
    try:
        print(f"🗑️ Destroying: {collection_name}")
        Collection(name=collection_name).destroy()
        cleanup_results.append(f"✅ {collection_name} - Successfully destroyed")
    except Exception as e:
        cleanup_results.append(f"ℹ️ {collection_name} - {e}")

print("\n📊 CLEANUP SUMMARY:")
for result in cleanup_results:
    print(f"  {result}")

print(f"\n🎯 CSV NOTEBOOK COMPLETE!")
print("📚 What we demonstrated:")
print("  ✅ Local CSV file processing with BasicIngestor")
print("  ✅ Multi-cloud CSV integration (S3, Azure Blob, GCP)")
print("  ✅ Structured data processing and business intelligence queries")
print("  ✅ Proper collection lifecycle management (create → use → destroy)")
print("  ✅ Complete resource cleanup to prevent memory leaks")
print("\n🔄 All CSV collections destroyed - system ready for next workflow!")

🧹 Comprehensive CSV Collection Cleanup Management...
🔄 This includes local and all cloud storage collections

🗑️ Destroying: csv_chocolate_sales_content_ingestor
Collection csv_chocolate_sales_content_ingestor initialized for use.
Collection destroy request is accepted and in progress
🗑️ Destroying: csv_s3_chocolate_sales_content
Collection csv_s3_chocolate_sales_content initialized for use.
Collection destroy request is accepted and in progress
🗑️ Destroying: csv_azure_chocolate_sales_content
Collection csv_azure_chocolate_sales_content initialized for use.
Collection destroy request is accepted and in progress
🗑️ Destroying: csv_gcp_chocolate_sales_content
Collection csv_gcp_chocolate_sales_content initialized for use.
Collection destroy request is accepted and in progress
🗑️ Destroying: csv_doc_mgmt_demo
Collection csv_doc_mgmt_demo initialized for use.
Collection destroy request is accepted and in progress

📊 CLEANUP SUMMARY:
  ✅ csv_chocolate_sales_content_ingestor - Successfully 

: 